# QuoridorAI — AlphaZero training (Stage 6A smoke run)

Run this notebook **in VS Code with a Colab kernel** (`Select Kernel → Colab → New Colab Server → GPU (T4)`; sign-in opens a browser once).
The local Windows files are **not** synced to the Colab runtime automatically, so the code travels as a verified bundle:

1. Local terminal (Windows): `npm run colab:bundle` → prints a `bundle_id` and writes `training/dist/quoridor_az_bundle.tar.gz`.
2. VS Code Explorer: right-click that file → **Upload to Colab** (it lands in `/content`).
   Alternative: copy it to Google Drive `MyDrive/QuoridorAI/bundles/` (cell 2 looks there too).
3. Optionally paste the printed id into `EXPECTED_BUNDLE_ID` below, then run the cells in order.

After every local code change: rebuild the bundle, upload it again and re-run cell 2 (it re-extracts and re-imports).
After a runtime restart: run cells 1–2 again (and cell 9 to reconnect Drive / resume the run).

Everything here is a **small smoke run** — no long training is started by this notebook.

## 1. Environment

In [ ]:
import importlib, os, platform, subprocess, sys, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
print('Python', platform.python_version(), '|', platform.platform())
print('Running on', 'Colab runtime' if IN_COLAB else 'local kernel')
import numpy as np
print('numpy', np.__version__)
try:
    import torch
    print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print('GPU', torch.cuda.get_device_name(0), f'| memory {total / 2**30:.1f} GiB total, {free / 2**30:.1f} GiB free')
except ImportError:
    print('torch is not installed in this kernel')
for mod in ('matplotlib', 'pytest'):
    print(mod, 'OK' if importlib.util.find_spec(mod) else 'missing')
if IN_COLAB:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

## 2. Project code (bundle → verified install)

In [ ]:
EXPECTED_BUNDLE_ID = ''  # paste the id printed by `npm run colab:bundle` to enforce an exact match
DRIVE_BUNDLE = Path('/content/drive/MyDrive/QuoridorAI/bundles/quoridor_az_bundle.tar.gz')

import hashlib, json, tarfile

def _verify(root):
    m = json.loads((root / 'BUNDLE_MANIFEST.json').read_text(encoding='utf-8'))
    bad = [f['path'] for f in m['files'] if hashlib.sha256((root / f['path']).read_bytes()).hexdigest() != f['sha256']]
    if bad:
        raise RuntimeError(f'files differ from manifest: {bad[:5]}')
    return m

if IN_COLAB:
    candidates = [Path('/content/quoridor_az_bundle.tar.gz'), DRIVE_BUNDLE]
    archive = next((p for p in candidates if p.exists()), None)
    if archive is None:
        raise FileNotFoundError('Upload training/dist/quoridor_az_bundle.tar.gz with "Upload to Colab" (or copy it to ' + str(DRIVE_BUNDLE) + ')')
    SRC = Path('/content/quoridor_src')
    if SRC.exists():
        import shutil
        shutil.rmtree(SRC)
    with tarfile.open(archive, 'r:gz') as tar:
        assert all(m.name.startswith('quoridor_src/') and '..' not in m.name for m in tar.getmembers())
        try:
            tar.extractall('/content', filter='data')
        except TypeError:
            tar.extractall('/content')
    manifest = _verify(SRC)
    print('bundle', manifest['bundle_id'], 'from', archive, '| git', (manifest['git_commit'] or 'n/a')[:7],
          '| uncommitted changes included' if manifest['git_dirty_training_files'] else '')
    if EXPECTED_BUNDLE_ID and manifest['bundle_id'] != EXPECTED_BUNDLE_ID:
        raise RuntimeError(f"bundle {manifest['bundle_id']} != expected {EXPECTED_BUNDLE_ID}: upload the current bundle")
    if importlib.util.find_spec('pytest') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
else:
    # Local kernel: use the repository directly (notebook lives in training/notebooks).
    SRC = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
    print('using local source tree', SRC)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
# Drop previously imported versions so a re-uploaded bundle takes effect without a restart.
for name in [m for m in sys.modules if m == 'quoridor_az' or m.startswith('quoridor_az.')]:
    del sys.modules[name]
import quoridor_az
from quoridor_az import engine as E, features as F
print('quoridor_az', quoridor_az.__version__, 'from', Path(quoridor_az.__file__).parent, '| feature encoding', F.FEATURE_VERSION)

## 3. Game environment (Python ⇄ TypeScript)

In [ ]:
s = E.create_initial_state()
print('legal actions at the start:', len(E.legal_actions(s)), '(3 pawn moves + 128 walls)')
print('planes', F.encode_state_planes(s).shape, '| canonical legal actions', int(F.canonical_legal_mask(s).sum()))
# Cross-language fixtures (generated from the TS engine) + rule tests.
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests/test_engine_fixtures.py', 'tests/test_engine_rules.py'], cwd=SRC, capture_output=True, text=True)
print(r.stdout[-600:] or r.stderr[-600:])
assert r.returncode == 0, 'cross-language tests failed'

## 4. Policy-value network

In [ ]:
import torch
from quoridor_az.network import NetConfig, PolicyValueNet, count_parameters, masked_log_softmax, pick_device

DEVICE = pick_device()
torch.manual_seed(0)
NET_CONFIG = NetConfig(channels=64, blocks=4)
net = PolicyValueNet(NET_CONFIG).to(DEVICE)
print('device', DEVICE, '| parameters', f'{count_parameters(net):,}', '|', net.metadata())

states = [E.create_initial_state(), E.create_state(pawns=((4, 4), (3, 4)), current_player=1)]
x = torch.from_numpy(np.stack([F.encode_state_planes(st) for st in states])).to(DEVICE)
mask = torch.from_numpy(np.stack([F.canonical_legal_mask(st) for st in states])).to(DEVICE)
logits, value = net(x)
p = masked_log_softmax(logits, mask).exp()
assert logits.shape == (2, 209) and value.shape == (2,)
assert torch.all(value.abs() <= 1) and torch.all(p[~mask] == 0) and torch.allclose(p.sum(1), torch.ones(2, device=DEVICE))
loss = -(p.detach() * masked_log_softmax(logits, mask).clamp_min(-1e4)).sum(1).mean() + value.pow(2).mean()
loss.backward()
assert all(torch.isfinite(q.grad).all() for q in net.parameters() if q.grad is not None)
print('forward/backward OK on', DEVICE, '| value', value.detach().cpu().numpy())

## 5. Neural MCTS

In [ ]:
from quoridor_az.evaluator import NeuralEvaluator
from quoridor_az.mcts import MctsConfig, Search

evaluator = NeuralEvaluator(net, DEVICE)
t0 = time.perf_counter()
search = Search(E.create_initial_state(), MctsConfig(num_simulations=200)).run(evaluator)
dt = time.perf_counter() - t0
print(f'{search.simulations} simulations in {dt:.2f}s ({search.simulations / dt:.0f}/s) | NN share {evaluator.stats.inference_seconds / dt:.0%}')
summary = search.summary()
print('root value', round(summary['root_value'], 3), '| net value', round(summary['net_value'], 3))
for c in summary['candidates']:
    print(f"  {E.format_action(E.decode_action(c['action'])):5s} N={c['visits']:4d} P={c['prior']:.3f} Q={c['q'] if c['q'] is None else round(c['q'], 3)}")
pi = search.visit_policy()
print('policy target sums to', pi.sum(), '| non-zero entries', int((pi > 0).sum()))

## 6. Self-play (small)

In [ ]:
from quoridor_az.selfplay import SelfPlayConfig, play_games

SP_CONFIG = SelfPlayConfig(mcts=MctsConfig(num_simulations=64, root_noise=True), max_plies=200, parallel_games=2, halted_policy='discard')
samples, records, sp_stats = play_games(NeuralEvaluator(net, DEVICE), SP_CONFIG, num_games=2, seed=0)
for r in records:
    print(r.outcome, 'winner', r.winner, '| plies', r.plies, f'| {r.seconds:.1f}s | samples kept {r.samples_kept}')
print(json.dumps(sp_stats.to_dict(), indent=1)[:800])

## 7. Training (smoke)

In [ ]:
from quoridor_az.replay import ReplayBuffer
from quoridor_az.train import TrainConfig, make_optimizer, train_step

buffer = ReplayBuffer(capacity=50_000)
buffer.add(samples)
if len(buffer) < 64:  # halted games are discarded, so top up with a draw-labelled game if needed
    extra, _, _ = play_games(NeuralEvaluator(net, DEVICE), SelfPlayConfig(mcts=MctsConfig(num_simulations=16), max_plies=80, parallel_games=2, halted_policy='draw'), 2, seed=1)
    buffer.add(extra)
print('buffer', len(buffer))
TRAIN_CONFIG = TrainConfig(batch_size=64, learning_rate=1e-3)
opt = make_optimizer(net, TRAIN_CONFIG)
rng = np.random.default_rng(0)
log = [train_step(net, opt, buffer.sample(TRAIN_CONFIG.batch_size, rng), TRAIN_CONFIG, DEVICE) for _ in range(100)]
assert all(l['finite'] for l in log), 'NaN/Inf in training'
print('first', {k: round(log[0][k], 4) for k in ('policy_loss', 'value_loss', 'grad_norm')})
print('last ', {k: round(log[-1][k], 4) for k in ('policy_loss', 'value_loss', 'grad_norm')})
print(f"step time {np.median([l['seconds'] for l in log]) * 1000:.1f} ms (batch {TRAIN_CONFIG.batch_size})")

## 8. Visualisation

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].plot([l['policy_loss'] for l in log], label='policy loss')
ax[0].plot([l['value_loss'] for l in log], label='value loss')
ax[0].set_xlabel('step')
ax[0].legend()
ax[0].set_title('Training losses')

batch = buffer.sample(256, rng)
with torch.no_grad():
    _, v = net(torch.from_numpy(batch['planes']).to(DEVICE))
ax[1].scatter(batch['value'] + rng.normal(0, 0.03, len(batch['value'])), v.cpu().numpy(), s=6, alpha=0.5)
ax[1].set_xlabel('target z')
ax[1].set_ylabel('predicted v')
ax[1].set_title('Value prediction')

top = summary['candidates']
ax[2].bar([E.format_action(E.decode_action(c['action'])) for c in top], [c['visits'] for c in top], label='MCTS visits')
ax2 = ax[2].twinx()
ax2.plot(range(len(top)), [c['prior'] for c in top], 'o-', color='tab:orange', label='prior')
ax[2].set_title('Root visits vs prior (start position)')
plt.tight_layout()
plt.show()

from quoridor_az.bench import measure
perf = measure(net, DEVICE, mcts_simulations=100, selfplay_games=2)
print(json.dumps(perf, indent=1, default=str))

## 9. Google Drive checkpoints

If `drive.mount` cannot open the sign-in from VS Code, run the command palette action **Colab: Mount Google Drive to Server…** instead.
Runs live in `MyDrive/QuoridorAI/runs/<RUN_NAME>`; re-running this cell on a new runtime resumes the latest valid checkpoint.

In [ ]:
from quoridor_az.pipeline import EvalConfig, RunConfig, Trainer

RUN_NAME = 'smoke-6a'
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        RUNS_ROOT = Path('/content/drive/MyDrive/QuoridorAI/runs')
    except Exception as e:  # noqa: BLE001
        print('Drive not mounted (', e, ') — using /content/runs (lost when the runtime ends)')
        RUNS_ROOT = Path('/content/runs')
else:
    RUNS_ROOT = SRC / 'runs'
RUN_DIR = RUNS_ROOT / RUN_NAME

SMOKE = RunConfig(
    seed=1, games_per_iteration=2, replay_capacity=20_000, net=NET_CONFIG,
    selfplay=SelfPlayConfig(mcts=MctsConfig(num_simulations=32, root_noise=True), max_plies=150, parallel_games=2, halted_policy='discard'),
    train=TrainConfig(batch_size=64, steps_per_iteration=20),
    eval=EvalConfig(pairs=1, simulations=16),
)
trainer = Trainer(RUN_DIR, SMOKE, DEVICE)
print('resumed' if trainer.resume() else 'new run', trainer.summary())
row = trainer.run_iteration()
print('checkpoint written:', sorted(p.name for p in (RUN_DIR / 'checkpoints').glob('*.pt'))[-1])

# Same output before saving and after loading into a fresh trainer.
probe = torch.from_numpy(F.encode_state_planes(E.create_initial_state())[None]).to(DEVICE)
trainer.net.eval()
with torch.no_grad():
    a = trainer.net(probe)
fresh = Trainer(RUN_DIR, SMOKE, DEVICE)
assert fresh.resume() and fresh.iteration == trainer.iteration
fresh.net.eval()
with torch.no_grad():
    b = fresh.net(probe)
assert torch.equal(a[0], b[0]) and torch.equal(a[1], b[1])
print('restore check OK at iteration', fresh.iteration, '| metrics rows', len(fresh.ckpt.read_metrics()))

## 10. Evaluation

In [ ]:
from quoridor_az.evaluate import GreedyDistancePlayer, MctsPlayer, RandomPlayer, play_match

current = MctsPlayer('latest', NeuralEvaluator(fresh.net, DEVICE), simulations=32)
torch.manual_seed(12345)
initial = MctsPlayer('initial (random weights)', NeuralEvaluator(PolicyValueNet(fresh.config.net), DEVICE), simulations=32)
for opponent in (initial, GreedyDistancePlayer(), RandomPlayer(seed=1)):
    res = play_match(current, opponent, pairs=1, seed=7)
    print(f'latest vs {opponent.name}:', res.to_dict())
# Same interface later: wrap any player (e.g. an ONNX export or a web AI port) with .choose(state) -> action index.

## 11. (Optional) ONNX export check

Preparation for Stage 6B (browser inference). Installs `onnx`, `onnxscript` and `onnxruntime` only when `EXPORT_ONNX = True`.
The file goes to the run directory; it is not copied into the web app in this stage.

In [ ]:
EXPORT_ONNX = False

if EXPORT_ONNX:
    if importlib.util.find_spec('onnxruntime') is None or importlib.util.find_spec('onnxscript') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxscript', 'onnxruntime'], check=True)
    import onnxruntime as ort
    from quoridor_az.network import export_onnx

    onnx_path = RUN_DIR / f'iter_{fresh.iteration:06d}.onnx'
    export_onnx(fresh.net, onnx_path)
    x = torch.from_numpy(np.stack([F.encode_state_planes(s) for s in [E.create_initial_state()] * 4]))
    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    onnx_logits, onnx_value = session.run(None, {'planes': x.numpy()})
    fresh.net.eval()
    with torch.no_grad():
        logits, value = fresh.net.cpu()(x)
    fresh.net.to(DEVICE)
    print(onnx_path.name, f'{onnx_path.stat().st_size / 1e6:.2f} MB')
    print('max |Δ logits|', float(np.abs(onnx_logits - logits.numpy()).max()), '| max |Δ value|', float(np.abs(onnx_value - value.numpy()).max()))
else:
    print('skipped (set EXPORT_ONNX = True)')